 # 1. Setup & Path Configuration

 This cell configures the workspace, silences expected future warnings,

 dynamically locates the dataset directory, and loads the raw data.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd

# Suppress the FutureWarnings regarding ffill/bfill downcasting
pd.set_option('future.no_silent_downcasting', True)

# 1. Dynamically locate the current directory
try:
    # If running as a standard .py script
    current_dir = Path(__file__).resolve().parent
except NameError:
    # Fallback if running inside a Jupyter Notebook (.ipynb)
    current_dir = Path.cwd()

# 2. Navigate up one level to 'stocks', then down into 'data'
data_dir = current_dir.parent / "data"

# 3. Define the specific file paths
df_ohlcv_path = data_dir / "df_OHLCV_stocks_etfs.parquet"
df_fed_path = data_dir / "High_Yield_Spread_T10Y2Y_Spread.csv"
df_indices_path = data_dir / "df_indices.parquet"  

# --- Load DataFrames ---

# 1. Load stocks parquet file
if df_ohlcv_path.exists():
    df_ohlcv = pd.read_parquet(df_ohlcv_path)
    print("Successfully loaded df_ohlcv!")
else:
    print(f"File not found: {df_ohlcv_path}")

# 2. Load High Yield Spread CSV file
if df_fed_path.exists():
    df_fed = pd.read_csv(df_fed_path)
    print("Successfully loaded df_fed!")
else:
    print(f"File not found: {df_fed_path}")

# 3. Load indices parquet file
if df_indices_path.exists():
    df_indices = pd.read_parquet(df_indices_path)  # Resolved name collision
    print("Successfully loaded df_indices!")
else:
    print(f"File not found: {df_indices_path}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Successfully loaded df_ohlcv!
Successfully loaded df_fed!
Successfully loaded df_indices!


 # 2. Feature Generation & Cleaning

 This cell configures your custom trading settings, executes the feature pipeline,

 and cleans/unstacks the output into wide-format matrices.

In [4]:
import pandas as pd
import numpy as np
import sys

# Ensure Python can find data_pipeline and core packages from the notebook directory
if str(current_dir) not in sys.path:
    sys.path.append(str(current_dir))

from data_pipeline import generate_features
from core.settings import TradingConfig

# Initialize configuration
config = TradingConfig()

print("Generating features_df and macro_df ... takes ~6 minutes")
print("[EXEC] Generating Features...")
features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv,
    df_indices=df_indices,
    df_fed=df_fed,
    config=config,
)

print("🚀 Unstacking Wide Matrices...")
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)
df_atrp_wide = features_df["ATRP"].unstack(level=0).reindex_like(df_close_wide)
df_trp_wide = features_df["TRP"].unstack(level=0).reindex_like(df_close_wide)

# Replace zeros with NaNs if enabled in the configurations
if config.handle_zeros_as_nan:
    df_close_wide = df_close_wide.replace(0, np.nan)

# Forward fill missing gaps up to threshold limit
df_close_wide = df_close_wide.ffill(limit=config.max_data_gap_ffill)
df_close_wide = df_close_wide.fillna(config.nan_price_replacement)

print(
    f"[OK] Pipeline Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)

Generating features_df and macro_df ... takes ~6 minutes
[EXEC] Generating Features...
[EXEC] Generating Decoupled Features (Benchmark: SPY)...
🚀 Unstacking Wide Matrices...
[OK] Pipeline Complete. Tickers: 1577, Days: 16214


 # 3. Save Processed Datasets

 Saves the final processed dataframes to the "processed" folder inside your data directory.

In [5]:
# Define dynamic destination directory
processed_dir = data_dir / "processed"
processed_dir.mkdir(
    parents=True, exist_ok=True
)  # Safely creates the directory if missing

# === SAVE TO PERSISTENCE SANDBOX ===
_fn = processed_dir / "features_df.parquet"
features_df.to_parquet(_fn, engine="pyarrow", index=True)
print(f"Saved: {_fn}")

_fn = processed_dir / "macro_df.parquet"
macro_df.to_parquet(_fn, engine="pyarrow", index=True)
print(f"Saved: {_fn}")

_fn = processed_dir / "df_close_wide.parquet"
df_close_wide.to_parquet(_fn, engine="pyarrow", index=True)
print(f"Saved: {_fn}")

_fn = processed_dir / "df_atrp_wide.parquet"
df_atrp_wide.to_parquet(_fn, engine="pyarrow", index=True)
print(f"Saved: {_fn}")

_fn = processed_dir / "df_trp_wide.parquet"
df_trp_wide.to_parquet(_fn, engine="pyarrow", index=True)
print(f"Saved: {_fn}")

Saved: c:\Users\ping\Files_win10\python\py311\stocks\data\processed\features_df.parquet
Saved: c:\Users\ping\Files_win10\python\py311\stocks\data\processed\macro_df.parquet
Saved: c:\Users\ping\Files_win10\python\py311\stocks\data\processed\df_close_wide.parquet
Saved: c:\Users\ping\Files_win10\python\py311\stocks\data\processed\df_atrp_wide.parquet
Saved: c:\Users\ping\Files_win10\python\py311\stocks\data\processed\df_trp_wide.parquet
